<a href="https://colab.research.google.com/github/muzazr/favera-model/blob/main/02_candidate_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:




















from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q \
    polars \
    pyarrow \
    pandas \
    numpy \
    scikit-learn \
    lightgbm \
    shap \
    matplotlib \
    joblib

In [3]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/favera/model/transaction-anomaly"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [4]:
import random
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [5]:
DATA_PATH = RAW_DATA_DIR / "momtsim_v1.csv"

lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=True
)

print(lazy_df.collect_schema())

sample_df = lazy_df.head(5).collect()
sample_df

row_count = lazy_df.select(pl.len()).collect().item()
print(f"jumlah transaksi: {row_count:,}")

Schema({'step': Int64, 'transactionType': String, 'amount': Float64, 'initiator': Int64, 'oldBalInitiator': Float64, 'newBalInitiator': Float64, 'recipient': String, 'oldBalRecipient': Float64, 'newBalRecipient': Float64, 'isFraud': Int64})
jumlah transaksi: 1,720,181


## 2. Memahami Struktur Dataset

In [6]:
from IPython.display import display, Markdown
import polars as pl

schema = lazy_df.collect_schema()
columns = schema.names()

schema_df = pl.DataFrame({
    "column": columns,
    "data_type": [str(schema[col]) for col in columns]
})

display(Markdown(
    f"Dataset memiliki **{row_count:,} baris** dan "
    f"**{len(columns)} kolom**."
))

display(schema_df)

Dataset memiliki **1,720,181 baris** dan **10 kolom**.

column,data_type
str,str
"""step""","""Int64"""
"""transactionType""","""String"""
"""amount""","""Float64"""
"""initiator""","""Int64"""
"""oldBalInitiator""","""Float64"""
"""newBalInitiator""","""Float64"""
"""recipient""","""String"""
"""oldBalRecipient""","""Float64"""
"""newBalRecipient""","""Float64"""


In [7]:
preview_df = lazy_df.head(10).collect()

display(preview_df)

EDA_SAMPLE_SIZE = 100_000

eda_df = lazy_df.head(EDA_SAMPLE_SIZE).collect(
    engine="streaming"
)

display(Markdown(
    f"Sampel EDA menggunakan **{eda_df.height:,} baris**."
))

step,transactionType,amount,initiator,oldBalInitiator,newBalInitiator,recipient,oldBalRecipient,newBalRecipient,isFraud
i64,str,f64,i64,f64,f64,str,f64,f64,i64
0,"""TRANSFER""",19824.96,4537027967639631,187712.18,167887.22,"""4875702729424478""",8.31,19833.27,1
0,"""PAYMENT""",598.97,4296267625767470,8.92,8.92,"""25-0000401""",0.0,0.0,0
0,"""PAYMENT""",545.85,4178224023847746,93.6,-452.25,"""13-0001587""",0.0,545.85,0
0,"""TRANSFER""",19847.01,4178224023847746,-452.25,-20299.26,"""4096920916696293""",4011.72,23858.74,1
0,"""PAYMENT""",546.89,4779013371563747,159148.76,158601.88,"""75-0003564""",0.0,546.89,0
0,"""TRANSFER""",16973.75,4779013371563747,158601.88,141628.12,"""4593247015015219""",54.33,17028.08,1
0,"""PAYMENT""",659.02,4479448076398077,261950.48,261291.46,"""11-0003866""",0.0,659.02,0
0,"""TRANSFER""",21103.99,4479448076398077,261291.46,240187.47,"""4649550475321678""",19.63,21123.63,1
0,"""PAYMENT""",648.89,4587942659105635,4808.94,4160.05,"""64-0008405""",0.0,648.89,0


Sampel EDA menggunakan **100,000 baris**.

In [8]:
null_summary = (
    lazy_df
    .select([
        pl.col(col).null_count().alias(col)
        for col in columns
    ])
    .collect(engine="streaming")
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"]
    )
    .with_columns(
        (
            pl.col("null_count") / row_count * 100
        ).round(4).alias("null_percentage")
    )
    .sort("null_count", descending=True)
)

columns_with_null = null_summary.filter(
    pl.col("null_count") > 0
)

if columns_with_null.height == 0:
    display(Markdown("**Tidak ditemukan missing value.**"))
else:
    display(columns_with_null)

**Tidak ditemukan missing value.**

In [9]:
sample_schema = eda_df.schema

numeric_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype.is_numeric()
]

string_cols = [
    col
    for col, dtype in sample_schema.items()
    if dtype == pl.String
]

print("Kolom numerik:", numeric_cols)
print("Kolom teks:", string_cols)



Kolom numerik: ['step', 'amount', 'initiator', 'oldBalInitiator', 'newBalInitiator', 'oldBalRecipient', 'newBalRecipient', 'isFraud']
Kolom teks: ['transactionType', 'recipient']


In [10]:
if numeric_cols:
    numeric_summary = eda_df.select(
        numeric_cols
    ).describe()

    display(numeric_summary)
else:
    display(Markdown(
        "**Tidak ditemukan kolom numerik pada hasil parsing awal.**"
    ))

statistic,step,amount,initiator,oldBalInitiator,newBalInitiator,oldBalRecipient,newBalRecipient,isFraud
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",5.56527,56655.657033,4.5010e15,287459.142625,332396.509199,7835.952872,12129.0505,0.11091
"""std""",2.983908,80062.006032,2.9027e14,569433.351835,601407.309418,37099.060757,42482.275094,0.314022
"""min""",0.0,1.36,4.0001e15,-199997.11,-199997.11,-149882.02,-135727.98,0.0
"""25%""",3.0,598.6,4.2472e15,18.44,20.4,534.48,1159.69,0.0
"""50%""",7.0,17572.83,4.5055e15,37481.59,83645.61,1735.43,2377.61,0.0
"""75%""",8.0,91043.64,4.7535e15,340451.41,425667.37,3303.1,4703.7,0.0
"""max""",9.0,1308624.7,4.9999e15,5.5312e6,5662492.3,1.5432e6,2.6180e6,1.0


In [11]:
if string_cols:
    unique_summary = pl.DataFrame({
        "column": string_cols,
        "unique_count_sample": [
            eda_df[col].n_unique()
            for col in string_cols
        ]
    }).sort(
        "unique_count_sample",
        descending=True
    )

    display(unique_summary)

column,unique_count_sample
str,i64
"""recipient""",10090
"""transactionType""",5


In [12]:
duplicate_count_sample = int(
    eda_df.is_duplicated().sum()
)

duplicate_percentage = (
    duplicate_count_sample / eda_df.height * 100
)

display(Markdown(
    f"""
**Duplikasi identik pada sampel:**
- Jumlah: **{duplicate_count_sample:,} baris**
- Persentase: **{duplicate_percentage:.4f}%**
"""
))


**Duplikasi identik pada sampel:**
- Jumlah: **0 baris**
- Persentase: **0.0000%**


In [13]:
target_keywords = [
    "fraud",
    "label",
    "class",
    "target",
    "anomaly"
]

time_keywords = [
    "date",
    "time",
    "timestamp",
    "step"
]

target_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in target_keywords)
]

time_candidates = [
    col for col in columns
    if any(keyword in col.lower() for keyword in time_keywords)
]

display(Markdown(
    f"""
### Kandidat Kolom Penting

**Kandidat label:** `{target_candidates}`

**Kandidat waktu:** `{time_candidates}`
"""
))


### Kandidat Kolom Penting

**Kandidat label:** `['isFraud']`

**Kandidat waktu:** `['step']`


In [14]:
for target_col in target_candidates:
    target_distribution = (
        lazy_df
        .group_by(target_col)
        .agg(
            pl.len().alias("count")
        )
        .with_columns(
            (
                pl.col("count") / row_count * 100
            ).round(4).alias("percentage")
        )
        .sort("count", descending=True)
        .collect(engine="streaming")
    )

    display(Markdown(
        f"### Distribusi `{target_col}`"
    ))
    display(target_distribution)

### Distribusi `isFraud`

isFraud,count,percentage
i64,u32,f64
0,1544663,89.7965
1,175518,10.2035


In [15]:
import re


def normalize_column_name(name: str) -> str:
    """Mengubah nama kolom menjadi format snake_case."""
    name = name.strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_").lower()


rename_map = {
    col: normalize_column_name(col)
    for col in columns
}

clean_lf = lazy_df.rename(rename_map)

display(
    pl.DataFrame({
        "original_column": list(rename_map.keys()),
        "clean_column": list(rename_map.values())
    })
)

original_column,clean_column
str,str
"""step""","""step"""
"""transactionType""","""transaction_type"""
"""amount""","""amount"""
"""initiator""","""initiator"""
"""oldBalInitiator""","""old_bal_initiator"""
"""newBalInitiator""","""new_bal_initiator"""
"""recipient""","""recipient"""
"""oldBalRecipient""","""old_bal_recipient"""
"""newBalRecipient""","""new_bal_recipient"""


In [16]:
clean_schema = clean_lf.collect_schema()

clean_string_cols = [
    col
    for col, dtype in clean_schema.items()
    if dtype == pl.String
]

NULL_TOKENS = [
    "",
    "na",
    "n/a",
    "null",
    "none",
    "-"
]

clean_lf = clean_lf.with_columns([
    pl.when(
        pl.col(col)
        .str.strip_chars()
        .str.to_lowercase()
        .is_in(NULL_TOKENS)
    )
    .then(None)
    .otherwise(
        pl.col(col).str.strip_chars()
    )
    .alias(col)
    for col in clean_string_cols
])

display(Markdown(
    f"Kolom teks yang dibersihkan: **{len(clean_string_cols)} kolom**."
))

Kolom teks yang dibersihkan: **2 kolom**.

In [17]:
STAGE_1_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_stage1_clean.parquet"
)

clean_lf.sink_parquet(
    str(STAGE_1_PATH),
    compression="zstd"
)

display(Markdown(
    f"""
### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`{STAGE_1_PATH}`
"""
))

clean_lf = pl.scan_parquet(
    STAGE_1_PATH
)


### Data Cleaning Tahap Awal Selesai

File tersimpan di:

`/content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_stage1_clean.parquet`


In [18]:
import polars as pl
from IPython.display import display, Markdown

# Kolom utama berdasarkan dataset kamu
TIME_COL = "step"
TARGET_COL = "is_fraud"

# Pastikan kolom wajib tersedia
required_columns = [
    "step",
    "transaction_type",
    "amount",
    "initiator",
    "old_bal_initiator",
    "new_bal_initiator",
    "recipient",
    "old_bal_recipient",
    "new_bal_recipient",
    "is_fraud",
]

available_columns = clean_lf.collect_schema().names()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

print("Semua kolom wajib tersedia.")

Semua kolom wajib tersedia.


In [19]:
# ============================================================
# FEATURE ENGINEERING AWAL
# ============================================================

feature_lf = (
    clean_lf
    .with_columns([
        # Tipe data dibuat konsisten
        pl.col("step").cast(pl.Int64),
        pl.col("amount").cast(pl.Float64),
        pl.col("initiator").cast(pl.String),
        pl.col("recipient").cast(pl.String),
        pl.col("is_fraud").cast(pl.Int8).alias("target"),

        pl.col("transaction_type")
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase(),
    ])
    .filter(
        pl.col("step").is_not_null() &
        pl.col("amount").is_not_null() &
        pl.col("target").is_not_null()
    )
    .with_columns([
        # Mengurangi efek nominal yang sangat ekstrem
        pl.col("amount")
        .clip(lower_bound=0)
        .log1p()
        .alias("amount_log1p"),

        # Perubahan saldo aktual
        (
            pl.col("new_bal_initiator") -
            pl.col("old_bal_initiator")
        ).alias("initiator_balance_change"),

        (
            pl.col("new_bal_recipient") -
            pl.col("old_bal_recipient")
        ).alias("recipient_balance_change"),

        # Rasio nominal terhadap saldo awal
        (
            pl.col("amount") /
            (pl.col("old_bal_initiator").abs() + 1)
        ).alias("amount_to_initiator_balance_ratio"),

        (
            pl.col("amount") /
            (pl.col("old_bal_recipient").abs() + 1)
        ).alias("amount_to_recipient_balance_ratio"),

        # Seberapa besar saldo pengirim berkurang
        (
            (
                pl.col("old_bal_initiator") -
                pl.col("new_bal_initiator")
            )
            / (pl.col("old_bal_initiator").abs() + 1)
        ).alias("initiator_balance_drop_ratio"),

        # Penanda kondisi saldo tertentu
        (
            pl.col("old_bal_initiator") <= 0
        ).cast(pl.Int8).alias("initiator_old_balance_non_positive"),

        (
            pl.col("new_bal_initiator") < 0
        ).cast(pl.Int8).alias("initiator_new_balance_negative"),

        (
            pl.col("old_bal_recipient") == 0
        ).cast(pl.Int8).alias("recipient_old_balance_zero"),
    ])
)

In [20]:
feature_lf = feature_lf.with_columns([
    # Saldo pengirim yang secara sederhana diharapkan
    (
        pl.col("old_bal_initiator") -
        pl.col("amount")
    ).alias("expected_new_bal_initiator"),

    # Saldo penerima yang secara sederhana diharapkan
    (
        pl.col("old_bal_recipient") +
        pl.col("amount")
    ).alias("expected_new_bal_recipient"),
])

feature_lf = feature_lf.with_columns([
    # Selisih antara saldo aktual dan saldo yang diharapkan
    (
        pl.col("new_bal_initiator") -
        pl.col("expected_new_bal_initiator")
    ).abs().alias("initiator_balance_error"),

    (
        pl.col("new_bal_recipient") -
        pl.col("expected_new_bal_recipient")
    ).abs().alias("recipient_balance_error"),
])

In [21]:
import numpy as np
import polars as pl

feature_lf = feature_lf.with_columns([
    # Jam relatif dalam sehari: 0 sampai 23
    (pl.col("step") % 24)
    .cast(pl.Int8)
    .alias("hour_of_day"),

    # Indeks hari sejak simulasi dimulai: 0 sampai 29
    (pl.col("step") // 24)
    .cast(pl.Int16)
    .alias("day_index"),
])

feature_lf = feature_lf.with_columns([
    # Transaksi malam, sementara didefinisikan pukul 22.00 sampai 04.59
    (
        (pl.col("hour_of_day") >= 22) |
        (pl.col("hour_of_day") < 5)
    )
    .cast(pl.Int8)
    .alias("is_night"),

    # Jam bersifat siklik. Jam 23 dekat dengan jam 0.
    (
        2 * np.pi * pl.col("hour_of_day") / 24
    )
    .sin()
    .alias("hour_sin"),

    (
        2 * np.pi * pl.col("hour_of_day") / 24
    )
    .cos()
    .alias("hour_cos"),
])

In [22]:
import gc
import json
import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

SEED = 42

# Target eksperimen candidate V2
TARGET_PR_AUC = 0.50
IDEAL_PR_AUC = 0.55
TARGET_PRECISION = 0.65
TARGET_RECALL = 0.80
TARGET_FPR = 0.10

In [23]:
def build_operating_point_table(
    y_true,
    probabilities,
):
    """
    Membuat seluruh operating point berdasarkan kemungkinan threshold.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.int8,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if len(y_true) != len(probabilities):
        raise ValueError(
            "Jumlah target dan probabilitas tidak sama."
        )

    # Skor tertinggi diletakkan lebih dahulu.
    order = np.argsort(
        -probabilities,
        kind="mergesort",
    )

    sorted_target = y_true[order]
    sorted_probability = probabilities[order]

    true_positive = np.cumsum(
        sorted_target
    )

    false_positive = np.cumsum(
        1 - sorted_target
    )

    total_positive = int(
        sorted_target.sum()
    )

    total_negative = (
        len(sorted_target)
        - total_positive
    )

    false_negative = (
        total_positive
        - true_positive
    )

    true_negative = (
        total_negative
        - false_positive
    )

    precision = (
        true_positive /
        np.maximum(
            true_positive + false_positive,
            1,
        )
    )

    recall = (
        true_positive /
        max(total_positive, 1)
    )

    false_positive_rate = (
        false_positive /
        max(total_negative, 1)
    )

    f1 = (
        2 * precision * recall /
        np.maximum(
            precision + recall,
            1e-12,
        )
    )

    # Ambil posisi terakhir untuk probabilitas yang sama.
    distinct_indices = np.r_[
        np.flatnonzero(
            sorted_probability[:-1]
            != sorted_probability[1:]
        ),
        len(sorted_probability) - 1,
    ]

    return pd.DataFrame({
        "threshold": (
            sorted_probability[
                distinct_indices
            ]
        ),
        "precision": (
            precision[
                distinct_indices
            ]
        ),
        "recall": (
            recall[
                distinct_indices
            ]
        ),
        "f1": (
            f1[
                distinct_indices
            ]
        ),
        "false_positive_rate": (
            false_positive_rate[
                distinct_indices
            ]
        ),
        "true_positive": (
            true_positive[
                distinct_indices
            ]
        ),
        "false_positive": (
            false_positive[
                distinct_indices
            ]
        ),
        "false_negative": (
            false_negative[
                distinct_indices
            ]
        ),
        "true_negative": (
            true_negative[
                distinct_indices
            ]
        ),
    })

In [24]:
def evaluate_model_target(
    model_name,
    y_true,
    probabilities,
):
    """
    Memilih threshold yang memenuhi target produk.

    Jika belum ada threshold yang memenuhi seluruh target,
    dipilih titik dengan precision terbaik sambil menjaga recall >= 80%.
    """
    operating_points = (
        build_operating_point_table(
            y_true,
            probabilities,
        )
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities,
    )

    feasible_points = operating_points[
        (
            operating_points["precision"]
            >= TARGET_PRECISION
        )
        &
        (
            operating_points["recall"]
            >= TARGET_RECALL
        )
        &
        (
            operating_points[
                "false_positive_rate"
            ]
            <= TARGET_FPR
        )
    ]

    if not feasible_points.empty:
        selected = (
            feasible_points
            .sort_values(
                [
                    "f1",
                    "precision",
                    "recall",
                ],
                ascending=False,
            )
            .iloc[0]
        )

        status = "TARGET ACHIEVED"

    else:
        recall_guardrail = operating_points[
            operating_points["recall"]
            >= TARGET_RECALL
        ]

        if recall_guardrail.empty:
            selected = (
                operating_points
                .sort_values(
                    "recall",
                    ascending=False,
                )
                .iloc[0]
            )
        else:
            selected = (
                recall_guardrail
                .sort_values(
                    [
                        "precision",
                        "false_positive_rate",
                        "f1",
                    ],
                    ascending=[
                        False,
                        True,
                        False,
                    ],
                )
                .iloc[0]
            )

        status = "TARGET NOT ACHIEVED"

    summary = {
        "model": model_name,
        "status": status,
        "pr_auc": float(pr_auc),
        "threshold": float(
            selected["threshold"]
        ),
        "precision": float(
            selected["precision"]
        ),
        "recall": float(
            selected["recall"]
        ),
        "f1": float(
            selected["f1"]
        ),
        "false_positive_rate": float(
            selected["false_positive_rate"]
        ),
        "true_positive": int(
            selected["true_positive"]
        ),
        "false_positive": int(
            selected["false_positive"]
        ),
        "false_negative": int(
            selected["false_negative"]
        ),
        "true_negative": int(
            selected["true_negative"]
        ),
    }

    return (
        pd.DataFrame([summary]),
        operating_points,
    )

In [25]:
BASELINE_PREDICTION_PATH = (
    OUTPUT_DIR /
    "lightgbm_baseline_v1_validation.parquet"
)

BASELINE_METADATA_PATH = (
    OUTPUT_DIR /
    "lightgbm_baseline_v1_metadata.json"
)

if not BASELINE_PREDICTION_PATH.exists():
    raise FileNotFoundError(
        "Baseline validation output belum ditemukan. "
        "Jalankan kode penyimpanan pada notebook baseline V1."
    )

baseline_validation_output = (
    pd.read_parquet(
        BASELINE_PREDICTION_PATH
    )
)

with open(
    BASELINE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    baseline_metadata = json.load(file)

baseline_y_validation = (
    baseline_validation_output["target"]
    .astype("int8")
    .to_numpy()
)

baseline_validation_probabilities = (
    baseline_validation_output[
        "probability"
    ]
    .to_numpy()
)

print("Metadata baseline:")
display(
    pd.DataFrame([
        baseline_metadata
    ])
)

Metadata baseline:


,model_version,f1_threshold,pr_auc
0,lightgbm_baseline_v1,0.709935,0.342034


In [26]:
(
    baseline_target_summary,
    baseline_operating_points,
) = evaluate_model_target(
    model_name="LightGBM Baseline V1",
    y_true=baseline_y_validation,
    probabilities=(
        baseline_validation_probabilities
    ),
)

display(baseline_target_summary)

,model,status,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Baseline V1,TARGET NOT ACHIEVED,0.342034,0.795496,0.308677,0.801281,0.445669,0.303094,21270,47637,5275,109532


In [27]:
validation_prevalence = (
    baseline_y_validation.mean()
)

required_fpr = (
    validation_prevalence
    * TARGET_RECALL
    * (
        (1 / TARGET_PRECISION)
        - 1
    )
    / (
        1 - validation_prevalence
    )
)

print(
    "Validation prevalence:",
    round(
        validation_prevalence,
        6,
    )
)

print(
    "Required maximum FPR:",
    round(
        required_fpr,
        6,
    )
)

Validation prevalence: 0.144491
Required maximum FPR: 0.072755


In [28]:
baseline_diagnostic_df = pd.DataFrame({
    "target": (
        baseline_validation_output[
            "target"
        ]
        .astype("int8")
    ),
    "transaction_type": (
        baseline_validation_output[
            "transaction_type"
        ]
        .astype(str)
    ),
    "probability": (
        baseline_validation_output[
            "probability"
        ]
    ),
})

baseline_selected_threshold = float(
    baseline_target_summary
    .iloc[0]["threshold"]
)

baseline_diagnostic_df["prediction"] = (
    baseline_diagnostic_df[
        "probability"
    ]
    >= baseline_selected_threshold
).astype("int8")

baseline_diagnostic_df["true_positive"] = (
    (
        baseline_diagnostic_df["target"] == 1
    )
    &
    (
        baseline_diagnostic_df[
            "prediction"
        ] == 1
    )
).astype("int8")

baseline_diagnostic_df["false_positive"] = (
    (
        baseline_diagnostic_df["target"] == 0
    )
    &
    (
        baseline_diagnostic_df[
            "prediction"
        ] == 1
    )
).astype("int8")

In [29]:
baseline_type_summary = (
    baseline_diagnostic_df
    .groupby(
        "transaction_type",
        observed=True,
    )
    .agg(
        transaction_count=(
            "target",
            "size",
        ),
        fraud_count=(
            "target",
            "sum",
        ),
        fraud_rate=(
            "target",
            "mean",
        ),
        average_score=(
            "probability",
            "mean",
        ),
        alert_count=(
            "prediction",
            "sum",
        ),
        true_positive=(
            "true_positive",
            "sum",
        ),
        false_positive=(
            "false_positive",
            "sum",
        ),
    )
    .reset_index()
)

baseline_type_summary["alert_rate"] = (
    baseline_type_summary[
        "alert_count"
    ]
    /
    baseline_type_summary[
        "transaction_count"
    ]
)

baseline_type_summary[
    "alert_precision"
] = (
    baseline_type_summary[
        "true_positive"
    ]
    /
    baseline_type_summary[
        "alert_count"
    ].clip(lower=1)
)

display(
    baseline_type_summary
    .sort_values(
        "fraud_rate",
        ascending=False,
    )
)

,transaction_type,transaction_count,fraud_count,fraud_rate,average_score,alert_count,true_positive,false_positive,alert_rate,alert_precision
3,TRANSFER,86524,26545,0.306793,0.799831,68907,21270,47637,0.796392,0.308677
0,DEBIT,81,0,0.000000,0.005197,0,0,0,0.000000,0.000000
1,DEPOSIT,6602,0,0.000000,0.005210,0,0,0,0.000000,0.000000
2,PAYMENT,89451,0,0.000000,0.005216,0,0,0,0.000000,0.000000
4,WITHDRAWAL,1056,0,0.000000,0.005691,0,0,0,0.000000,0.000000


In [30]:
required_history_columns = [
    "step",
    "initiator",
    "recipient",
    "amount",
]

available_feature_columns = (
    feature_lf
    .collect_schema()
    .names()
)

missing_history_columns = [
    column
    for column in required_history_columns
    if column not in available_feature_columns
]

if missing_history_columns:
    raise ValueError(
        "Kolom histori tidak ditemukan: "
        f"{missing_history_columns}"
    )

# Materialisasi hanya empat kolom yang diperlukan.
# Ini mencegah seluruh feature_lf masuk RAM sekaligus.
base_history_df = (
    feature_lf
    .select(
        required_history_columns
    )
    .collect(
        engine="streaming"
    )
)

print(
    "History source shape:",
    base_history_df.shape,
)

initiator_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "initiator_tx_count_step"
        ),

        pl.col("amount")
        .sum()
        .alias(
            "initiator_amount_sum_step"
        ),

        pl.col("amount")
        .pow(2)
        .sum()
        .alias(
            "initiator_amount_sq_sum_step"
        ),

        pl.col("amount")
        .max()
        .alias(
            "initiator_amount_max_step"
        ),
    ])
    .sort([
        "initiator",
        "step",
    ])
)

print(
    "Initiator-step rows:",
    initiator_step_df.height,
)

History source shape: (1720181, 4)
Initiator-step rows: 558223


In [31]:
initiator_history_df = (
    initiator_step_df
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_tx_count_step"
            )
        )
        .alias(
            "initiator_tx_count_prior"
        ),

        (
            pl.col(
                "initiator_amount_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sum_step"
            )
        )
        .alias(
            "initiator_amount_sum_prior"
        ),

        (
            pl.col(
                "initiator_amount_sq_sum_step"
            )
            .cum_sum()
            .over(
                "initiator",
                order_by="step",
            )
            -
            pl.col(
                "initiator_amount_sq_sum_step"
            )
        )
        .alias(
            "initiator_amount_sq_sum_prior"
        ),

        pl.col(
            "initiator_amount_max_step"
        )
        .cum_max()
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_amount_max_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            "initiator",
            order_by="step",
        )
        .alias(
            "initiator_previous_step"
        ),
    ])
)

In [32]:
initiator_6h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="6i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_6h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_6h"
        ),
    ])
)

In [33]:
initiator_24h_df = (
    initiator_step_df
    .select([
        "initiator",
        "step",
        "initiator_tx_count_step",
        "initiator_amount_sum_step",
    ])
    .sort([
        "initiator",
        "step",
    ])
    .rolling(
        index_column="step",
        period="24i",
        closed="left",
        group_by="initiator",
    )
    .agg([
        pl.col(
            "initiator_tx_count_step"
        )
        .sum()
        .alias(
            "initiator_tx_count_24h"
        ),

        pl.col(
            "initiator_amount_sum_step"
        )
        .sum()
        .alias(
            "initiator_amount_sum_24h"
        ),
    ])
)

In [34]:
initiator_history_df = (
    initiator_history_df
    .join(
        initiator_6h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        initiator_24h_df,
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
)

In [35]:
recipient_step_df = (
    base_history_df
    .group_by([
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "recipient_tx_count_step"
        ),
    ])
    .sort([
        "recipient",
        "step",
    ])
)

In [36]:
recipient_history_df = (
    recipient_step_df
    .with_columns([
        (
            pl.col(
                "recipient_tx_count_step"
            )
            .cum_sum()
            .over(
                "recipient",
                order_by="step",
            )
            -
            pl.col(
                "recipient_tx_count_step"
            )
        )
        .alias(
            "recipient_tx_count_prior"
        ),
    ])
    .select([
        "recipient",
        "step",
        "recipient_tx_count_prior",
    ])
)

In [37]:
pair_step_df = (
    base_history_df
    .group_by([
        "initiator",
        "recipient",
        "step",
    ])
    .agg([
        pl.len()
        .alias(
            "pair_tx_count_step"
        ),
    ])
    .sort([
        "initiator",
        "recipient",
        "step",
    ])
)

pair_history_df = (
    pair_step_df
    .with_columns([
        (
            pl.col(
                "pair_tx_count_step"
            )
            .cum_sum()
            .over(
                [
                    "initiator",
                    "recipient",
                ],
                order_by="step",
            )
            -
            pl.col(
                "pair_tx_count_step"
            )
        )
        .alias(
            "pair_tx_count_prior"
        ),

        pl.col("step")
        .shift(1)
        .over(
            [
                "initiator",
                "recipient",
            ],
            order_by="step",
        )
        .alias(
            "pair_previous_step"
        ),
    ])
    .select([
        "initiator",
        "recipient",
        "step",
        "pair_tx_count_prior",
        "pair_previous_step",
    ])
)

In [38]:
behavioral_feature_lf = (
    feature_lf
    .join(
        initiator_history_df.lazy(),
        on=[
            "initiator",
            "step",
        ],
        how="left",
    )
    .join(
        recipient_history_df.lazy(),
        on=[
            "recipient",
            "step",
        ],
        how="left",
    )
    .join(
        pair_history_df.lazy(),
        on=[
            "initiator",
            "recipient",
            "step",
        ],
        how="left",
    )
)

In [39]:
history_zero_columns = [
    "initiator_tx_count_prior",
    "initiator_amount_sum_prior",
    "initiator_amount_sq_sum_prior",
    "initiator_amount_max_prior",
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "recipient_tx_count_prior",
    "pair_tx_count_prior",
]

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.col(column)
        .fill_null(0)
        .alias(column)
        for column in history_zero_columns
    ])
)

In [40]:
behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        (
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .cast(pl.Int8)
        .alias(
            "has_initiator_history"
        ),

        (
            pl.col(
                "pair_tx_count_prior"
            ) == 0
        )
        .cast(pl.Int8)
        .alias(
            "is_new_recipient"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col(
                "initiator_amount_sum_prior"
            )
            /
            pl.col(
                "initiator_tx_count_prior"
            )
        )
        .otherwise(0)
        .alias(
            "initiator_amount_mean_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col(
                    "initiator_amount_sq_sum_prior"
                )
                /
                pl.col(
                    "initiator_tx_count_prior"
                )
            )
            -
            pl.col(
                "initiator_amount_mean_prior"
            ).pow(2)
        )
        .otherwise(0)
        .clip(lower_bound=0)
        .alias(
            "initiator_amount_variance_prior"
        ),
    ])
    .with_columns([
        pl.col(
            "initiator_amount_variance_prior"
        )
        .sqrt()
        .alias(
            "initiator_amount_std_prior"
        ),
    ])
)

behavioral_feature_lf = (
    behavioral_feature_lf
    .with_columns([
        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_mean_prior"
                ).abs()
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_mean"
        ),

        pl.when(
            pl.col(
                "initiator_tx_count_prior"
            ) > 1
        )
        .then(
            (
                pl.col("amount")
                -
                pl.col(
                    "initiator_amount_mean_prior"
                )
            )
            /
            (
                pl.col(
                    "initiator_amount_std_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_zscore_initiator"
        ),

        pl.when(
            pl.col(
                "initiator_amount_max_prior"
            ) > 0
        )
        .then(
            pl.col("amount")
            /
            (
                pl.col(
                    "initiator_amount_max_prior"
                )
                + 1
            )
        )
        .otherwise(0)
        .alias(
            "amount_vs_initiator_max"
        ),

        pl.when(
            pl.col(
                "initiator_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "initiator_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_initiator_transaction"
        ),

        pl.when(
            pl.col(
                "pair_previous_step"
            ).is_not_null()
        )
        .then(
            pl.col("step")
            -
            pl.col(
                "pair_previous_step"
            )
        )
        .otherwise(-1)
        .alias(
            "steps_since_pair_transaction"
        ),
    ])
)

In [41]:
behavioral_preview = (
    behavioral_feature_lf
    .select([
        "step",
        "initiator",
        "recipient",
        "amount",
        "initiator_tx_count_prior",
        "initiator_amount_mean_prior",
        "amount_vs_initiator_mean",
        "initiator_tx_count_6h",
        "initiator_tx_count_24h",
        "recipient_tx_count_prior",
        "pair_tx_count_prior",
        "is_new_recipient",
        "target",
    ])
    .head(10)
    .collect()
)

display(behavioral_preview)

step,initiator,recipient,amount,initiator_tx_count_prior,initiator_amount_mean_prior,amount_vs_initiator_mean,initiator_tx_count_6h,initiator_tx_count_24h,recipient_tx_count_prior,pair_tx_count_prior,is_new_recipient,target
i64,str,str,f64,u32,f64,f64,u32,u32,u32,u32,i8,i8
0,"""4537027967639631""","""4875702729424478""",19824.96,0,0.0,0.0,0,0,0,0,1,1
0,"""4296267625767470""","""25-0000401""",598.97,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""13-0001587""",545.85,0,0.0,0.0,0,0,0,0,1,0
0,"""4178224023847746""","""4096920916696293""",19847.01,0,0.0,0.0,0,0,0,0,1,1
0,"""4779013371563747""","""75-0003564""",546.89,0,0.0,0.0,0,0,0,0,1,0
0,"""4779013371563747""","""4593247015015219""",16973.75,0,0.0,0.0,0,0,0,0,1,1
0,"""4479448076398077""","""11-0003866""",659.02,0,0.0,0.0,0,0,0,0,1,0
0,"""4479448076398077""","""4649550475321678""",21103.99,0,0.0,0.0,0,0,0,0,1,1
0,"""4587942659105635""","""64-0008405""",648.89,0,0.0,0.0,0,0,0,0,1,0


In [42]:
behavioral_row_count = (
    behavioral_feature_lf
    .select(
        pl.len()
    )
    .collect(
        engine="streaming"
    )
    .item()
)

print(
    "Rows before behavioral features:",
    row_count,
)

print(
    "Rows after behavioral features:",
    behavioral_row_count,
)

if behavioral_row_count != row_count:
    raise ValueError(
        "Jumlah baris berubah setelah join behavioral features."
    )

Rows before behavioral features: 1720181
Rows after behavioral features: 1720181


In [43]:
unique_steps_v2 = (
    behavioral_feature_lf
    .select("step")
    .unique()
    .sort("step")
    .collect(
        engine="streaming"
    )
    .get_column("step")
)

number_of_steps_v2 = len(
    unique_steps_v2
)

train_step_index_v2 = (
    int(
        number_of_steps_v2 * 0.70
    )
    - 1
)

validation_step_index_v2 = (
    int(
        number_of_steps_v2 * 0.85
    )
    - 1
)

train_end_step_v2 = (
    unique_steps_v2[
        train_step_index_v2
    ]
)

validation_end_step_v2 = (
    unique_steps_v2[
        validation_step_index_v2
    ]
)

print(
    "Jumlah step:",
    number_of_steps_v2,
)

print(
    "Train sampai step:",
    train_end_step_v2,
)

print(
    "Validation sampai step:",
    validation_end_step_v2,
)

print(
    "Test setelah step:",
    validation_end_step_v2,
)

Jumlah step: 144
Train sampai step: 99
Validation sampai step: 121
Test setelah step: 121


In [44]:
train_v2_lf = (
    behavioral_feature_lf
    .filter(
        pl.col("step")
        <= train_end_step_v2
    )
)

validation_v2_lf = (
    behavioral_feature_lf
    .filter(
        (
            pl.col("step")
            > train_end_step_v2
        )
        &
        (
            pl.col("step")
            <= validation_end_step_v2
        )
    )
)

test_v2_lf = (
    behavioral_feature_lf
    .filter(
        pl.col("step")
        > validation_end_step_v2
    )
)

In [45]:
def summarize_split_v2(
    split_name,
    split_lf,
):
    return (
        split_lf
        .select([
            pl.lit(
                split_name
            ).alias("split"),

            pl.len()
            .alias("row_count"),

            pl.col("step")
            .min()
            .alias("step_start"),

            pl.col("step")
            .max()
            .alias("step_end"),

            pl.col("target")
            .sum()
            .alias("fraud_count"),

            pl.col("target")
            .mean()
            .alias("fraud_rate"),
        ])
        .collect(
            engine="streaming"
        )
    )

split_summary_v2 = pl.concat([
    summarize_split_v2(
        "train",
        train_v2_lf,
    ),
    summarize_split_v2(
        "validation",
        validation_v2_lf,
    ),
    summarize_split_v2(
        "test",
        test_v2_lf,
    ),
])

display(split_summary_v2)

split,row_count,step_start,step_end,fraud_count,fraud_rate
str,u32,i64,i64,i64,f64
"""train""",1242432,0,99,122258,0.098402
"""validation""",183714,100,121,26545,0.144491
"""test""",294035,122,143,26715,0.090857


In [46]:
TRAIN_V2_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_train_features_v2.parquet"
)

VALIDATION_V2_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_validation_features_v2.parquet"
)

TEST_V2_PATH = (
    PROCESSED_DATA_DIR /
    "momtsim_test_features_v2.parquet"
)

In [47]:
train_v2_lf.sink_parquet(
    str(TRAIN_V2_PATH),
    compression="zstd",
)

validation_v2_lf.sink_parquet(
    str(VALIDATION_V2_PATH),
    compression="zstd",
)

test_v2_lf.sink_parquet(
    str(TEST_V2_PATH),
    compression="zstd",
)

print("Train V2:", TRAIN_V2_PATH)
print("Validation V2:", VALIDATION_V2_PATH)
print("Test V2:", TEST_V2_PATH)

Train V2: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v2.parquet
Validation V2: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v2.parquet
Test V2: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v2.parquet


In [48]:
del base_history_df
del initiator_step_df
del initiator_6h_df
del initiator_24h_df
del recipient_step_df
del pair_step_df

gc.collect()

print("Intermediate memory dibersihkan.")

Intermediate memory dibersihkan.


In [49]:
V2_NUMERIC_FEATURES = [
    # Transaction features
    "amount",
    "amount_log1p",
    "old_bal_initiator",
    "new_bal_initiator",
    "old_bal_recipient",
    "new_bal_recipient",
    "initiator_balance_change",
    "recipient_balance_change",
    "amount_to_initiator_balance_ratio",
    "amount_to_recipient_balance_ratio",
    "initiator_balance_drop_ratio",
    "initiator_old_balance_non_positive",
    "initiator_new_balance_negative",
    "recipient_old_balance_zero",

    # Time features
    "hour_of_day",
    "is_night",
    "hour_sin",
    "hour_cos",

    # Behavioral features
    "has_initiator_history",
    "initiator_tx_count_prior",
    "initiator_amount_mean_prior",
    "initiator_amount_std_prior",
    "initiator_amount_max_prior",
    "amount_vs_initiator_mean",
    "amount_zscore_initiator",
    "amount_vs_initiator_max",
    "initiator_tx_count_6h",
    "initiator_amount_sum_6h",
    "initiator_tx_count_24h",
    "initiator_amount_sum_24h",
    "steps_since_initiator_transaction",
    "recipient_tx_count_prior",
    "pair_tx_count_prior",
    "is_new_recipient",
    "steps_since_pair_transaction",
]

V2_CATEGORICAL_FEATURES = [
    "transaction_type",
]

V2_TARGET = "target"

In [50]:
def load_v2_split(path):
    selected_columns = (
        V2_NUMERIC_FEATURES
        + V2_CATEGORICAL_FEATURES
        + [V2_TARGET]
    )

    numeric_expressions = [
        pl.col(column)
        .cast(
            pl.Float32,
            strict=False,
        )
        .fill_nan(0)
        .fill_null(0)
        .alias(column)
        for column in V2_NUMERIC_FEATURES
    ]

    categorical_expressions = [
        pl.col(column)
        .cast(pl.String)
        .fill_null("UNKNOWN")
        .alias(column)
        for column
        in V2_CATEGORICAL_FEATURES
    ]

    return (
        pl.scan_parquet(
            str(path)
        )
        .select(
            selected_columns
        )
        .with_columns(
            numeric_expressions
            + categorical_expressions
            + [
                pl.col(V2_TARGET)
                .cast(pl.Int8)
                .alias(V2_TARGET)
            ]
        )
        .collect(
            engine="streaming"
        )
        .to_pandas()
    )

In [51]:
train_v2_df = load_v2_split(
    TRAIN_V2_PATH
)

validation_v2_df = load_v2_split(
    VALIDATION_V2_PATH
)

print(
    "Train V2 shape:",
    train_v2_df.shape,
)

print(
    "Validation V2 shape:",
    validation_v2_df.shape,
)

print(
    "Train V2 memory:",
    round(
        train_v2_df
        .memory_usage(
            deep=True
        )
        .sum()
        / 1024**2,
        2,
    ),
    "MB",
)

Train V2 shape: (1242432, 37)
Validation V2 shape: (183714, 37)
Train V2 memory: 233.99 MB


In [52]:
X_train_v2 = (
    train_v2_df[
        V2_NUMERIC_FEATURES
        + V2_CATEGORICAL_FEATURES
    ]
    .copy()
)

y_train_v2 = (
    train_v2_df[V2_TARGET]
    .astype("int8")
)

X_validation_v2 = (
    validation_v2_df[
        V2_NUMERIC_FEATURES
        + V2_CATEGORICAL_FEATURES
    ]
    .copy()
)

y_validation_v2 = (
    validation_v2_df[V2_TARGET]
    .astype("int8")
)

print(
    "Train fraud rate:",
    y_train_v2.mean(),
)

print(
    "Validation fraud rate:",
    y_validation_v2.mean(),
)

Train fraud rate: 0.09840216607428012
Validation fraud rate: 0.1444908934539556


In [53]:
transaction_type_categories_v2 = sorted(
    X_train_v2[
        "transaction_type"
    ]
    .dropna()
    .unique()
    .tolist()
)

transaction_type_dtype_v2 = (
    pd.CategoricalDtype(
        categories=(
            transaction_type_categories_v2
        )
    )
)

X_train_v2[
    "transaction_type"
] = (
    X_train_v2[
        "transaction_type"
    ]
    .astype(
        transaction_type_dtype_v2
    )
)

X_validation_v2[
    "transaction_type"
] = (
    X_validation_v2[
        "transaction_type"
    ]
    .astype(
        transaction_type_dtype_v2
    )
)

print(
    "Transaction categories:",
    transaction_type_categories_v2,
)

Transaction categories: ['DEBIT', 'DEPOSIT', 'PAYMENT', 'TRANSFER', 'WITHDRAWAL']


In [54]:
lightgbm_v2_model = LGBMClassifier(
    objective="binary",

    n_estimators=1500,
    learning_rate=0.05,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    class_weight=None,

    random_state=SEED,
    n_jobs=-1,
)

lightgbm_v2_model.fit(
    X_train_v2,
    y_train_v2,

    eval_set=[
        (
            X_validation_v2,
            y_validation_v2,
        )
    ],

    eval_metric=(
        "average_precision"
    ),

    categorical_feature=[
        "transaction_type"
    ],

    callbacks=[
        early_stopping(
            stopping_rounds=75,
            first_metric_only=True,
            verbose=True,
        ),
        log_evaluation(
            period=50
        ),
    ],
)

[LightGBM] [Info] Number of positive: 122258, number of negative: 1120174
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.474836 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6057
[LightGBM] [Info] Number of data points in the train set: 1242432, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.098402 -> initscore=-2.215106
[LightGBM] [Info] Start training from score -2.215106
Training until validation scores don't improve for 75 rounds
[50]	valid_0's average_precision: 0.357238	valid_0's binary_logloss: 0.291079
[100]	valid_0's average_precision: 0.356602	valid_0's binary_logloss: 0.287867
Early stopping, best iteration is:
[44]	valid_0's average_precision: 0.358543	valid_0's binary_logloss: 0.292171
Evaluated only: average_precision


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, min_child_samples=50,
               n_estimators=1500, n_jobs=-1, objective='binary',
               random_state=42, reg_alpha=0.1, reg_lambda=0.1, subsample=0.8,
               subsample_freq=1)

In [55]:
lightgbm_v2_validation_probabilities = (
    lightgbm_v2_model
    .predict_proba(
        X_validation_v2,
        num_iteration=(
            lightgbm_v2_model
            .best_iteration_
        ),
    )[:, 1]
)

print(
    "Best iteration:",
    lightgbm_v2_model.best_iteration_,
)

print(
    "Candidate V2 PR-AUC/AP:",
    average_precision_score(
        y_validation_v2,
        lightgbm_v2_validation_probabilities,
    ),
)

Best iteration: 44
Candidate V2 PR-AUC/AP: 0.3585434380921033


In [56]:
(
    v2_target_summary,
    v2_operating_points,
) = evaluate_model_target(
    model_name=(
        "LightGBM Candidate V2"
    ),
    y_true=y_validation_v2,
    probabilities=(
        lightgbm_v2_validation_probabilities
    ),
)

display(v2_target_summary)

,model,status,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Candidate V2,TARGET NOT ACHIEVED,0.358543,0.243964,0.310123,0.800829,0.447104,0.30088,21258,47289,5287,109880


In [57]:
baseline_vs_v2 = pd.concat(
    [
        baseline_target_summary,
        v2_target_summary,
    ],
    ignore_index=True,
)

comparison_columns = [
    "model",
    "status",
    "pr_auc",
    "threshold",
    "precision",
    "recall",
    "f1",
    "false_positive_rate",
    "true_positive",
    "false_positive",
    "false_negative",
    "true_negative",
]

display(
    baseline_vs_v2[
        comparison_columns
    ]
)

,model,status,pr_auc,threshold,precision,recall,f1,false_positive_rate,true_positive,false_positive,false_negative,true_negative
0,LightGBM Baseline V1,TARGET NOT ACHIEVED,0.342034,0.795496,0.308677,0.801281,0.445669,0.303094,21270,47637,5275,109532
1,LightGBM Candidate V2,TARGET NOT ACHIEVED,0.358543,0.243964,0.310123,0.800829,0.447104,0.300880,21258,47289,5287,109880


In [58]:
baseline_result = (
    baseline_target_summary
    .iloc[0]
)

v2_result = (
    v2_target_summary
    .iloc[0]
)

improvement_summary = pd.DataFrame([{
    "pr_auc_change": (
        v2_result["pr_auc"]
        - baseline_result["pr_auc"]
    ),
    "precision_change": (
        v2_result["precision"]
        - baseline_result["precision"]
    ),
    "recall_change": (
        v2_result["recall"]
        - baseline_result["recall"]
    ),
    "fpr_change": (
        v2_result[
            "false_positive_rate"
        ]
        -
        baseline_result[
            "false_positive_rate"
        ]
    ),
    "false_positive_reduction": (
        baseline_result[
            "false_positive"
        ]
        -
        v2_result[
            "false_positive"
        ]
    ),
}])

display(improvement_summary)

,pr_auc_change,precision_change,recall_change,fpr_change,false_positive_reduction
0,0.01651,0.001446,-0.000452,-0.002214,348


In [59]:
v2_diagnostic_df = pd.DataFrame({
    "target": (
        y_validation_v2
        .to_numpy()
    ),
    "transaction_type": (
        X_validation_v2[
            "transaction_type"
        ]
        .astype(str)
        .to_numpy()
    ),
    "probability": (
        lightgbm_v2_validation_probabilities
    ),
})

v2_selected_threshold = float(
    v2_target_summary
    .iloc[0]["threshold"]
)

v2_diagnostic_df["prediction"] = (
    v2_diagnostic_df[
        "probability"
    ]
    >= v2_selected_threshold
).astype("int8")

v2_diagnostic_df["true_positive"] = (
    (
        v2_diagnostic_df["target"] == 1
    )
    &
    (
        v2_diagnostic_df[
            "prediction"
        ] == 1
    )
).astype("int8")

v2_diagnostic_df["false_positive"] = (
    (
        v2_diagnostic_df["target"] == 0
    )
    &
    (
        v2_diagnostic_df[
            "prediction"
        ] == 1
    )
).astype("int8")

In [60]:
v2_type_summary = (
    v2_diagnostic_df
    .groupby(
        "transaction_type",
        observed=True,
    )
    .agg(
        transaction_count=(
            "target",
            "size",
        ),
        fraud_count=(
            "target",
            "sum",
        ),
        fraud_rate=(
            "target",
            "mean",
        ),
        average_score=(
            "probability",
            "mean",
        ),
        alert_count=(
            "prediction",
            "sum",
        ),
        true_positive=(
            "true_positive",
            "sum",
        ),
        false_positive=(
            "false_positive",
            "sum",
        ),
    )
    .reset_index()
)

v2_type_summary["alert_rate"] = (
    v2_type_summary[
        "alert_count"
    ]
    /
    v2_type_summary[
        "transaction_count"
    ]
)

v2_type_summary[
    "alert_precision"
] = (
    v2_type_summary[
        "true_positive"
    ]
    /
    v2_type_summary[
        "alert_count"
    ].clip(lower=1)
)

display(
    v2_type_summary
    .sort_values(
        "fraud_rate",
        ascending=False,
    )
)

,transaction_type,transaction_count,fraud_count,fraud_rate,average_score,alert_count,true_positive,false_positive,alert_rate,alert_precision
3,TRANSFER,86524,26545,0.306793,0.275478,68547,21258,47289,0.792231,0.310123
0,DEBIT,81,0,0.000000,0.010892,0,0,0,0.000000,0.000000
1,DEPOSIT,6602,0,0.000000,0.010895,0,0,0,0.000000,0.000000
2,PAYMENT,89451,0,0.000000,0.010895,0,0,0,0.000000,0.000000
4,WITHDRAWAL,1056,0,0.000000,0.010952,0,0,0,0.000000,0.000000


In [61]:
feature_importance_v2 = pd.DataFrame({
    "feature": (
        lightgbm_v2_model
        .booster_
        .feature_name()
    ),
    "importance_gain": (
        lightgbm_v2_model
        .booster_
        .feature_importance(
            importance_type="gain"
        )
    ),
    "importance_split": (
        lightgbm_v2_model
        .booster_
        .feature_importance(
            importance_type="split"
        )
    ),
})

feature_importance_v2[
    "gain_percentage"
] = (
    feature_importance_v2[
        "importance_gain"
    ]
    /
    feature_importance_v2[
        "importance_gain"
    ].sum()
    * 100
)

feature_importance_v2 = (
    feature_importance_v2
    .sort_values(
        "importance_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    feature_importance_v2.head(20)
)

,feature,importance_gain,importance_split,gain_percentage
0,transaction_type,2.038628e+06,36,75.304491
1,recipient_balance_change,3.486532e+05,67,12.878838
2,initiator_amount_max_prior,9.803721e+04,74,3.621378
3,new_bal_initiator,6.084729e+04,102,2.247627
4,amount,4.546214e+04,78,1.679318
5,initiator_balance_change,2.481621e+04,24,0.916681
6,amount_vs_initiator_mean,1.658082e+04,96,0.612476
7,initiator_new_balance_negative,1.585769e+04,12,0.585764
8,old_bal_initiator,1.270430e+04,81,0.469282
9,recipient_tx_count_prior,1.090576e+04,139,0.402846


In [62]:
V2_MODEL_PATH = (
    MODEL_DIR /
    "lightgbm_candidate_v2.txt"
)

lightgbm_v2_model.booster_.save_model(
    str(V2_MODEL_PATH)
)

In [63]:
V2_PREDICTION_PATH = (
    OUTPUT_DIR /
    "lightgbm_candidate_v2_validation.parquet"
)

v2_validation_output = pd.DataFrame({
    "target": (
        y_validation_v2
        .to_numpy()
    ),
    "transaction_type": (
        X_validation_v2[
            "transaction_type"
        ]
        .astype(str)
        .to_numpy()
    ),
    "probability": (
        lightgbm_v2_validation_probabilities
    ),
})

v2_validation_output.to_parquet(
    V2_PREDICTION_PATH,
    index=False,
)

In [64]:
selected_v2_result = (
    v2_target_summary
    .iloc[0]
    .to_dict()
)

v2_metadata = {
    "model_version": (
        "lightgbm_candidate_v2"
    ),
    "best_iteration": int(
        lightgbm_v2_model
        .best_iteration_
    ),
    "threshold": float(
        selected_v2_result[
            "threshold"
        ]
    ),
    "pr_auc": float(
        selected_v2_result[
            "pr_auc"
        ]
    ),
    "precision": float(
        selected_v2_result[
            "precision"
        ]
    ),
    "recall": float(
        selected_v2_result[
            "recall"
        ]
    ),
    "f1": float(
        selected_v2_result[
            "f1"
        ]
    ),
    "false_positive_rate": float(
        selected_v2_result[
            "false_positive_rate"
        ]
    ),
    "numeric_features": (
        V2_NUMERIC_FEATURES
    ),
    "categorical_features": (
        V2_CATEGORICAL_FEATURES
    ),
    "transaction_type_categories": (
        transaction_type_categories_v2
    ),
}

V2_METADATA_PATH = (
    MODEL_DIR /
    "lightgbm_candidate_v2_metadata.json"
)

with open(
    V2_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        v2_metadata,
        file,
        indent=2,
    )

FEATURE_IMPORTANCE_PATH = (
    OUTPUT_DIR /
    "lightgbm_candidate_v2_feature_importance.csv"
)

feature_importance_v2.to_csv(
    FEATURE_IMPORTANCE_PATH,
    index=False,
)

print("Model:", V2_MODEL_PATH)
print("Metadata:", V2_METADATA_PATH)
print("Predictions:", V2_PREDICTION_PATH)
print(
    "Feature importance:",
    FEATURE_IMPORTANCE_PATH,
)

Model: /content/drive/MyDrive/favera/model/transaction-anomaly/models/lightgbm_candidate_v2.txt
Metadata: /content/drive/MyDrive/favera/model/transaction-anomaly/models/lightgbm_candidate_v2_metadata.json
Predictions: /content/drive/MyDrive/favera/model/transaction-anomaly/outputs/lightgbm_candidate_v2_validation.parquet
Feature importance: /content/drive/MyDrive/favera/model/transaction-anomaly/outputs/lightgbm_candidate_v2_feature_importance.csv


In [65]:
# ============================================================
# FAVERA MODEL 1 - UNIVERSAL DIAGNOSTIC EXPORT
# Tempel di cell paling bawah lalu jalankan
# ============================================================

import json
import os
import sys
import platform
import zipfile
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

EXPORT_DIR = Path("/content/favera_diagnostic_export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

report_lines = []

def add_title(title):
    report_lines.append("\n" + "=" * 80)
    report_lines.append(title)
    report_lines.append("=" * 80)

def add_text(text=""):
    report_lines.append(str(text))

def export_dataframe(name, dataframe, max_print_rows=100):
    if dataframe is None:
        return

    try:
        if hasattr(dataframe, "to_pandas"):
            dataframe = dataframe.to_pandas()

        dataframe = pd.DataFrame(dataframe)

        csv_path = EXPORT_DIR / f"{name}.csv"
        dataframe.to_csv(csv_path, index=False)

        add_title(name.upper())
        add_text(dataframe.head(max_print_rows).to_string(index=False))
        add_text(f"\nShape: {dataframe.shape}")
        add_text(f"File: {csv_path}")

    except Exception as error:
        add_text(f"Gagal mengekspor {name}: {error}")

def variable_exists(name):
    return name in globals() and globals()[name] is not None

# ============================================================
# 1. INFORMASI ENVIRONMENT
# ============================================================

add_title("INFORMASI EKSEKUSI")

add_text(f"Waktu ekspor: {datetime.now().isoformat()}")
add_text(f"Python: {sys.version}")
add_text(f"Platform: {platform.platform()}")
add_text(f"Working directory: {os.getcwd()}")

try:
    import sklearn
    add_text(f"scikit-learn: {sklearn.__version__}")
except Exception:
    pass

try:
    import lightgbm
    add_text(f"LightGBM: {lightgbm.__version__}")
except Exception:
    pass

try:
    import polars
    add_text(f"Polars: {polars.__version__}")
except Exception:
    pass

# ============================================================
# 2. DATASET DAN SPLIT
# ============================================================

add_title("RINGKASAN DATASET")

for variable_name in [
    "row_count",
    "TRAIN_MAX_STEP",
    "VALIDATION_MAX_STEP",
    "TEST_MAX_STEP",
    "TRAIN_V2_PATH",
    "VALIDATION_V2_PATH",
    "TEST_V2_PATH",
]:
    if variable_exists(variable_name):
        add_text(f"{variable_name}: {globals()[variable_name]}")

for variable_name in [
    "split_summary",
    "split_stats",
    "dataset_summary",
    "target_distribution",
    "null_summary",
]:
    if variable_exists(variable_name):
        export_dataframe(
            variable_name,
            globals()[variable_name],
        )

# ============================================================
# 3. IDENTIFIKASI DATA VALIDATION
# ============================================================

y_true = None
probabilities = None
selected_threshold = None
model_name = "Tidak teridentifikasi"

if variable_exists("y_validation_v2"):
    y_true = np.asarray(
        globals()["y_validation_v2"]
    ).astype(int)

    model_name = "LightGBM Candidate V2"

    if variable_exists("lightgbm_v2_validation_probabilities"):
        probabilities = np.asarray(
            globals()["lightgbm_v2_validation_probabilities"]
        )

    if variable_exists("v2_selected_threshold"):
        selected_threshold = float(
            globals()["v2_selected_threshold"]
        )

    elif variable_exists("v2_target_summary"):
        try:
            selected_threshold = float(
                globals()["v2_target_summary"]
                .iloc[0]["threshold"]
            )
        except Exception:
            pass

elif variable_exists("y_validation"):
    y_true = np.asarray(
        globals()["y_validation"]
    ).astype(int)

    model_name = "Baseline Model"

    probability_candidates = [
        "lightgbm_validation_probabilities",
        "validation_probabilities",
        "y_validation_probability",
        "y_validation_probabilities",
    ]

    for candidate in probability_candidates:
        if variable_exists(candidate):
            probabilities = np.asarray(
                globals()[candidate]
            )
            break

    threshold_candidates = [
        "selected_threshold",
        "lightgbm_selected_threshold",
        "baseline_selected_threshold",
    ]

    for candidate in threshold_candidates:
        if variable_exists(candidate):
            selected_threshold = float(
                globals()[candidate]
            )
            break

# ============================================================
# 4. METRIK VALIDATION
# ============================================================

add_title("HASIL EVALUASI VALIDATION")
add_text(f"Model: {model_name}")

if y_true is not None and probabilities is not None:

    if selected_threshold is None:
        selected_threshold = 0.5

    predictions = (
        probabilities >= selected_threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    metrics = {
        "model": model_name,
        "validation_rows": len(y_true),
        "positive_count": int(y_true.sum()),
        "negative_count": int((y_true == 0).sum()),
        "positive_rate": float(y_true.mean()),
        "threshold": float(selected_threshold),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_negative": int(tn),
        "false_positive_rate": float(
            fp / max(fp + tn, 1)
        ),
        "alert_rate": float(
            predictions.mean()
        ),
    }

    try:
        metrics["roc_auc"] = float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        )
    except Exception:
        metrics["roc_auc"] = None

    metrics_df = pd.DataFrame([metrics])
    export_dataframe(
        "validation_metrics",
        metrics_df,
    )

    add_title("CLASSIFICATION REPORT")
    add_text(
        classification_report(
            y_true,
            predictions,
            digits=6,
            zero_division=0,
        )
    )

    confusion_df = pd.DataFrame(
        [
            {
                "actual": "normal",
                "predicted_normal": int(tn),
                "predicted_anomaly": int(fp),
            },
            {
                "actual": "fraud",
                "predicted_normal": int(fn),
                "predicted_anomaly": int(tp),
            },
        ]
    )

    export_dataframe(
        "confusion_matrix",
        confusion_df,
    )

    prediction_output = pd.DataFrame({
        "target": y_true,
        "probability": probabilities,
        "prediction": predictions,
    })

    export_dataframe(
        "validation_predictions",
        prediction_output,
        max_print_rows=30,
    )

else:
    add_text(
        "Variabel target atau probability validation tidak ditemukan."
    )

# ============================================================
# 5. TABEL HASIL YANG SUDAH ADA
# ============================================================

for variable_name in [
    "logistic_target_summary",
    "lightgbm_target_summary",
    "baseline_target_summary",
    "v2_target_summary",
    "baseline_vs_v2",
    "improvement_summary",
    "v2_operating_points",
    "operating_points",
]:
    if variable_exists(variable_name):
        export_dataframe(
            variable_name,
            globals()[variable_name],
        )

# ============================================================
# 6. DIAGNOSTIK TRANSACTION TYPE
# ============================================================

for variable_name in [
    "v2_type_summary",
    "type_summary",
    "transaction_type_summary",
]:
    if variable_exists(variable_name):
        export_dataframe(
            variable_name,
            globals()[variable_name],
        )

validation_dataframe = None

if variable_exists("validation_v2_df"):
    validation_dataframe = globals()["validation_v2_df"]

elif variable_exists("validation_df"):
    validation_dataframe = globals()["validation_df"]

if validation_dataframe is not None:
    try:
        if hasattr(validation_dataframe, "to_pandas"):
            validation_dataframe = (
                validation_dataframe.to_pandas()
            )

        validation_dataframe = pd.DataFrame(
            validation_dataframe
        )

        add_title("VALIDATION DATAFRAME")
        add_text(
            f"Shape: {validation_dataframe.shape}"
        )
        add_text(
            f"Columns: {validation_dataframe.columns.tolist()}"
        )

        target_column = None

        for candidate in [
            "target",
            "is_fraud",
            "label",
        ]:
            if candidate in validation_dataframe.columns:
                target_column = candidate
                break

        if (
            "transaction_type"
            in validation_dataframe.columns
            and target_column is not None
        ):
            type_fraud_table = (
                validation_dataframe
                .groupby(
                    "transaction_type",
                    dropna=False,
                )[target_column]
                .agg(
                    transaction_count="size",
                    fraud_count="sum",
                    fraud_rate="mean",
                )
                .reset_index()
                .sort_values(
                    "fraud_rate",
                    ascending=False,
                )
            )

            export_dataframe(
                "transaction_type_fraud_distribution",
                type_fraud_table,
            )

        history_columns = [
            column
            for column
            in validation_dataframe.columns
            if any(
                keyword in column.lower()
                for keyword in [
                    "history",
                    "prior",
                    "recipient",
                    "frequency",
                    "count_",
                    "new_recipient",
                    "time_since",
                ]
            )
        ]

        if history_columns:
            history_summary = (
                validation_dataframe[
                    history_columns
                ]
                .describe(
                    include="all"
                )
                .transpose()
                .reset_index()
                .rename(
                    columns={"index": "feature"}
                )
            )

            export_dataframe(
                "history_feature_summary",
                history_summary,
            )

    except Exception as error:
        add_text(
            f"Gagal menganalisis validation dataframe: {error}"
        )

# ============================================================
# 7. FEATURE IMPORTANCE
# ============================================================

for variable_name in [
    "feature_importance_v2",
    "feature_importance",
    "lightgbm_feature_importance",
]:
    if variable_exists(variable_name):
        export_dataframe(
            variable_name,
            globals()[variable_name],
        )

# Jika tabel belum dibuat, ambil langsung dari model
for model_variable in [
    "lightgbm_v2_model",
    "lightgbm_model",
]:
    if variable_exists(model_variable):
        try:
            model = globals()[model_variable]

            booster = (
                model.booster_
                if hasattr(model, "booster_")
                else model
            )

            importance_df = pd.DataFrame({
                "feature": booster.feature_name(),
                "importance_gain": (
                    booster.feature_importance(
                        importance_type="gain"
                    )
                ),
                "importance_split": (
                    booster.feature_importance(
                        importance_type="split"
                    )
                ),
            })

            total_gain = (
                importance_df[
                    "importance_gain"
                ].sum()
            )

            importance_df[
                "gain_percentage"
            ] = np.where(
                total_gain > 0,
                (
                    importance_df[
                        "importance_gain"
                    ]
                    / total_gain
                    * 100
                ),
                0,
            )

            importance_df = (
                importance_df
                .sort_values(
                    "importance_gain",
                    ascending=False,
                )
                .reset_index(drop=True)
            )

            export_dataframe(
                f"{model_variable}_feature_importance",
                importance_df,
            )

            add_title(
                f"{model_variable.upper()} PARAMETERS"
            )

            if hasattr(model, "get_params"):
                add_text(
                    json.dumps(
                        model.get_params(),
                        indent=2,
                        default=str,
                    )
                )

            if hasattr(model, "best_iteration_"):
                add_text(
                    f"Best iteration: "
                    f"{model.best_iteration_}"
                )

        except Exception as error:
            add_text(
                f"Gagal mengambil feature importance "
                f"{model_variable}: {error}"
            )

# ============================================================
# 8. DAFTAR VARIABEL PENTING YANG TERSEDIA
# ============================================================

add_title("VARIABEL NOTEBOOK YANG TERSEDIA")

important_keywords = [
    "model",
    "metric",
    "summary",
    "threshold",
    "probab",
    "prediction",
    "feature",
    "validation",
    "test",
    "train",
    "split",
]

available_variables = sorted([
    name
    for name in globals().keys()
    if not name.startswith("_")
    and any(
        keyword in name.lower()
        for keyword in important_keywords
    )
])

for name in available_variables:
    try:
        value = globals()[name]
        add_text(
            f"{name}: {type(value).__name__}"
        )
    except Exception:
        pass

# ============================================================
# 9. SIMPAN REPORT
# ============================================================

REPORT_PATH = EXPORT_DIR / "FULL_DIAGNOSTIC_REPORT.txt"

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "\n".join(report_lines)
    )

# Print seluruh laporan agar bisa langsung dicopy
print("\n".join(report_lines))

# ============================================================
# 10. ZIP DAN DOWNLOAD
# ============================================================

ZIP_PATH = Path(
    "/content/favera_model_diagnostic_export.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as zip_file:

    for file_path in EXPORT_DIR.rglob("*"):
        if file_path.is_file():
            zip_file.write(
                file_path,
                arcname=file_path.relative_to(
                    EXPORT_DIR
                ),
            )

print("\n" + "=" * 80)
print("EXPORT SELESAI")
print("=" * 80)
print(f"Report: {REPORT_PATH}")
print(f"ZIP: {ZIP_PATH}")
print(
    "Kirim file ZIP tersebut beserta notebook "
    ".ipynb yang sudah memiliki output."
)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as error:
    print(
        "Download otomatis gagal. "
        "Unduh manual melalui panel Files Colab:",
        error,
    )


INFORMASI EKSEKUSI
Waktu ekspor: 2026-08-01T11:20:17.535088
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content
scikit-learn: 1.6.1
LightGBM: 4.6.0
Polars: 1.35.2

RINGKASAN DATASET
row_count: 1720181
TRAIN_V2_PATH: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_train_features_v2.parquet
VALIDATION_V2_PATH: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_validation_features_v2.parquet
TEST_V2_PATH: /content/drive/MyDrive/favera/model/transaction-anomaly/data/processed/momtsim_test_features_v2.parquet

TARGET_DISTRIBUTION
 isFraud   count  percentage
       0 1544663     89.7965
       1  175518     10.2035

Shape: (2, 3)
File: /content/favera_diagnostic_export/target_distribution.csv

NULL_SUMMARY
         column  null_count  null_percentage
           step           0              0.0
transactionType           0              0.0
         amo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>